**Proyecto 1: Análisis de señales ECG y detección de arritmias**

**Integrantes:** Paula García · Valentina Morales


**Fecha de entrega:**

---

 **1.1 Diagramas de flujo**

Artículo 1. Gadaleta et al. (2023), *npj Digital Medicine*
Predicción de fibrilación auricular a 2 semanas a partir de ECG de una derivación sin arritmias.

![Diagrama artículo 1](diagrama_articulo1.png)

Artículo 2. Zheng et al. (2020), *Scientific Reports*
Clasificación multietapa de arritmias (base Chapman).

![Diagrama artículo 2](diagrama_articulo2.png)

Artículo 3. Liu et al. (2024), *Heliyon*
Clasificación de arritmias con CNN sobre ECG de una derivación, sin detección del QRS.

![Diagrama artículo 3](diagrama_articulo3.png)

---

**1.2 Explicación de las metodologías con nuestras palabras**

**Artículo 1:** La fibrilación auricular es una arritmia en la que las aurículas laten de manera descontrolada. Es difícil de detectar con métodos tradicionales porque puede aparecer y desaparecer de forma súbita e impredecible, por lo que ni un electrocardiograma de diez segundos, ni siquiera un Holter de veinticuatro horas garantizan capturar un episodio, ya que no se sabe cuándo va a ocurrir.

Frente a ese problema, el artículo propone un modelo que recibe ventanas de ECG de diez minutos y predice si la persona podría tener un episodio de fibrilación auricular en las siguientes dos semanas, aunque el ECG analizado no muestre ningún signo de la arritmia. Se usan ventanas de diez minutos por un tema de costo computacional, ya que es mucho más manejable para los modelos procesar esa cantidad de datos que la señal completa de veinticuatro horas de una sola vez.

El primer modelo se entrena con cada ventana de forma independiente y extrae ciento veintiocho parámetros propios de la morfología de la señal, que no corresponden a fórmulas definidas por médicos o estadísticos, sino a patrones que el modelo considera relevantes durante su entrenamiento. Además, para cada ventana también se calculan estadísticos ya conocidos, como la variabilidad del ritmo cardíaco y la cantidad de latidos ectópicos. En total, un día se divide en ciento cuarenta y cuatro ventanas, y por cada una se obtiene un paquete que combina los ciento veintiocho parámetros de la red con los estadísticos tradicionales. Esa secuencia completa de ventanas se le pasa al segundo modelo, que la procesa en orden temporal y decide qué tan en riesgo está la persona de sufrir fibrilación auricular.
Para evaluar el desempeño se usa el *AUC*, un índice que indica qué tan bien distingue el modelo entre una persona que va a tener fibrilación auricular y una que no, y que va de 0.5, equivalente al azar, a 1, que sería un modelo perfecto.
Usando solamente edad y sexo, el *AUC* es de 0.67; agregando la variabilidad del ritmo cardíaco sube a 0.74; y utilizando todos los datos disponibles, incluyendo los ciento veintiocho parámetros de la red neuronal y los estadísticos tradicionales, llega a 0.80.

Sin embargo, de la cohorte estudiada solo el 4.3% de las personas tuvo realmente un episodio de fibrilación auricular durante esas dos semanas. Esa baja prevalencia hace que, aunque el modelo tenga una sensibilidad del 0.80, su precisión sea naturalmente baja, de apenas 0.09, lo que significa que se generan muchos falsos positivos: de cada cien alertas de riesgo alto, solo nueve corresponden a casos reales de fibrilación auricular.

**Artículo 2:** Las arritmias cardíacas se diagnostican con un electrocardiograma, pero interpretarlo depende de que un cardiólogo lo revise, lo que toma tiempo y limita la detección a gran escala. El artículo propone un método automático que, a partir de un ECG, clasifica el ritmo que el paciente tiene en ese momento (no predice lo que pueda pasar en el futuro) en cuatro grupos: bradicardia sinusal, fibrilación o aleteo auricular, taquicardias supraventriculares y ritmo sinusal.

Se usó una base de datos de más de cuarenta mil pacientes, con ECG de doce derivaciones de diez segundos, donde los ritmos fueron diagnosticados por cardiólogos. Originalmente había once ritmos distintos, y se agruparon en cuatro según su relevancia clínica, porque algunos tenían muy pocos casos para entrenar un modelo.

Primero se limpia la señal en tres pasos: un filtro pasa-bajas que elimina el ruido de alta frecuencia, un método llamado LOESS robusto que corrige el movimiento lento de la línea base, por debajo de 0.5 Hz, y un tercer método, Non Local Means, que quita el ruido que queda. Después se detectan los picos y valles de la señal, que se clasifican en complejos QRS, picos que no son QRS y valles, y de cada uno se miden la altura, el ancho y la prominencia. También se comparan estos elementos por pares y se calculan razones respecto al tiempo entre ellos.

El problema es que cada paciente tiene un número distinto de latidos, y por lo tanto de picos y valles, pero el modelo necesita que todos los pacientes tengan la misma cantidad de datos. Por eso cada variable se resume en un histograma de cien intervalos, que cuenta cuántas veces aparece cada rango de valores. Se usa un histograma y no un promedio porque el promedio pierde la forma de la distribución: dos pacientes pueden tener el mismo valor promedio pero comportamientos muy distintos, y el histograma sí lo captura.

Con esos datos se compararon diecinueve algoritmos de clasificación combinados con cinco estrategias, ajustando sus parámetros y validando con validación cruzada de diez partes. Los mejores fueron los árboles con gradient boosting. En el boosting los árboles se construyen uno tras otro, y cada uno intenta corregir los errores del anterior, a diferencia del bagging, donde los árboles se entrenan de forma independiente y se promedian. Con este método el *F1-score* fue de 0.988 en pacientes sin otras condiciones cardíacas y de 0.97 en pacientes que sí las tenían. El F1 combina la precisión y la sensibilidad, por lo que un valor alto indica que el modelo clasifica bien sin sacrificar ninguna de las dos.

También se probó el método en la base MIT-BIH y se obtuvo un *F1-score* de 0.992, aunque ahí la tarea fue clasificar latidos individuales y no ritmos, por lo que no es una validación del mismo problema. Otra limitación es que el artículo no aclara en cuál de los cuatro grupos queda la taquicardia sinusal, que representa más del 15% de los pacientes, y que la afirmación de desempeño comparable al de cardiólogos no se midió directamente contra ellos.

**Artículo 3:** Muchos métodos de clasificación de arritmias dependen de detectar primero el complejo QRS, para medir intervalos entre latidos y calcular rasgos a partir de ahí. El problema es que esa detección falla fácilmente cuando la señal tiene ruido o movimiento, que es lo que pasa con los dispositivos portátiles. El artículo propone un método que clasifica arritmias directamente sobre la señal, sin detectar el QRS, usando una red neuronal convolucional.

Se usan señales de una sola derivación de tres bases de datos de PhysioNet, con nueve ritmos cardíacos: ritmo sinusal normal, fibrilación auricular, aleteo auricular, Wolff-Parkinson-White, bloqueo Mobitz II, bradicardia sinusal, taquicardia ventricular, fibrilación ventricular y aleteo ventricular. Como las bases venían con frecuencias de muestreo distintas, todas se llevaron a 250 Hz. Luego las señales se cortaron en segmentos de cinco y de diez segundos, y para aumentar el número de muestras se usaron ventanas superpuestas, es decir, segmentos que comparten una parte de la señal con el siguiente.

La red recibe el segmento completo y tiene cuatro capas convolucionales, con max pooling entre ellas. La primera capa detecta patrones simples de la señal, y las siguientes los van combinando en estructuras cada vez más complejas. El max pooling reduce el tamaño de lo que se procesa y conserva lo más relevante. Al final, capas totalmente conectadas y una función softmax dan la probabilidad de que el segmento corresponda a cada uno de los nueve ritmos.

Con segmentos de cinco segundos la exactitud fue de 96.51%, y con diez segundos de 98.1%. Además, el modelo de cinco segundos se probó con una base distinta, CUDB, que no se usó para entrenar y en la que se emplearon los ritmos completos sin solapamiento, y ahí la exactitud bajó a 91.6%. Una de las confusiones más frecuentes fue clasificar fibrilación auricular como aleteo auricular.

Hay varias limitaciones. Con ventanas superpuestas puede haber fuga de datos: si una ventana queda en entrenamiento y otra, que comparte parte de la misma señal, queda en validación, el modelo estaría evaluándose con algo que ya vio casi igual, lo que inflaría la exactitud interna. El artículo no aclara cómo se dividieron los datos, así que esto es una posibilidad y no algo confirmado. Eso encaja con que la exactitud baje en la prueba externa.